# Getting started with Rerank endpoint

## Overview

Traditional keyword-based search often fails to capture semantic meaning and context, leading to irrelevant results. Cohere's rerank endpoint addresses this by reordering documents based on semantic relevance to a query, improving search accuracy across different languages and reasoning types.

This tutorial demonstrates how to use Cohere's rerank endpoint to improve document ranking through semantic similarity.

By the end, you will be able to implement semantic reranking for various use cases including multilingual queries, numeric comparisons, temporal filtering, and negation handling.

## What we'll cover
- Using the rerank endpoint
- Exploring response object structure
- Implementing multilingual reranking
- Handling numeric reasoning
- Processing temporal reasoning
- Managing negation queries
- Reranking PDF pages and extracting relevant passages with `relevant_spans`

## Using the Rerank endpoint

In [ ]:
import cohere
import os
import yaml
import json
from dotenv import load_dotenv
load_dotenv(override=True)

top_n = 3
MODEL = os.environ["RERANK_MODEL"]
co = cohere.ClientV2(
    base_url=os.environ["RERANK_BASE_URL"],
    api_key=os.environ["RERANK_API_KEY"],
    timeout=300,
)

def return_results(results, documents):
    for idx, result in enumerate(results.results):
        print(f"Rank: {idx+1}")
        print(f"Score: {result.relevance_score}")
        print(f"Document: {documents[result.index]}\n")
        
query = "Are there fitness-related perks?"

company_faqs = [
    "Reimbursing Travel Expenses: Easily manage your travel expenses by submitting them through our finance tool. Approvals are prompt and straightforward.",
    "Working from Abroad: Working remotely from another country is possible. Simply coordinate with your manager and ensure your availability during core hours.",
    "Health and Wellness Benefits: We care about your well-being and offer gym memberships, on-site yoga classes, and comprehensive health insurance.",
    "Performance Reviews Frequency: We conduct informal check-ins every quarter and formal performance reviews twice a year.",
]

results = co.rerank(
    model=MODEL, 
    query=query, 
    documents=company_faqs,
    top_n=top_n
)

print('*** This is the first rerank endpoint call.  If you get errors here, check your .env file for correctness.\n\n')
print(f"Query: {query}\n")
return_results(results, company_faqs)


## Multilingual reranking

In [ ]:
        
# Define the query
query = "피트니스 관련 혜택이 있나요?"  # Are there fitness benefits?

# Rerank the documents
results = co.rerank(
    model=MODEL,
    query=query,
    documents=company_faqs,
    top_n=top_n,
)

print(f"*** Query: {query}\n")
return_results(results, company_faqs)

## Numeric reasoning

In [ ]:
query = "standing desk shorter than 130 cm"

documents = [
    "Standing desk 125 cm, adjustable height",
    "Standing desk 100 cm, compact design",
    "Standing desk 135 cm, with cable tray",
    "Standing desk 140 cm, dual motor",
    "Standing desk 120 cm, white frame"
]

results = co.rerank(
    model=MODEL, 
    query=query, 
    documents=documents,
    top_n=len(documents)
)

print(f"*** Query: {query}\n")
return_results(results, documents)

## Temporal reasoning

In [ ]:
query = "list all my flights between May 1st and June 15th"

documents = [
    "Itinerary: Flight AA123, New York (JFK) to London (LHR), Departure: April 28, 2024, 7:00 PM, Seat 14A",
    "E-Ticket: Flight LH456, Paris (CDG) to Berlin (TXL), Departure: May 10, 2024, 9:30 AM, Seat 22C",
    "Booking Confirmation: Flight JL789, Tokyo (HND) to Seoul (ICN), Departure: May 25, 2024, 2:15 PM, Seat 18B",
    "Itinerary: Flight UA321, Los Angeles (LAX) to San Francisco (SFO), Departure: June 20, 2024, 11:00 AM, Seat 10D",
    "E-Ticket: Flight AZ654, Rome (FCO) to Madrid (MAD), Departure: June 5, 2024, 6:45 AM, Seat 7F",
]

results = co.rerank(
    model=MODEL, 
    query=query, 
    documents=documents,
    top_n=len(documents)
)

print(f"*** Query: {query}\n")
return_results(results, documents)


In [ ]:
# flights that occur before the specified month and not the month itself. 
query = "What flights do I have before May?"


results = co.rerank(
    model=MODEL, 
    query=query, 
    documents=documents,
    top_n=len(documents)
)

print(f"*** Query: {query}\n")
return_results(results, documents)

## Negation

In [ ]:
query = "salad without walnuts"

documents = [
    "Spinach salad with walnuts and cranberries",
    "Arugula salad with goat cheese and walnuts",
    "Greek salad with feta and olives",
    "Caesar salad with parmesan and croutons",
    "Tomato basil soup with garlic bread",
]

results = co.rerank(
    model=MODEL, 
    query=query, 
    documents=documents,
    top_n=len(documents)
)

print(f"*** Query: {query}\n")
return_results(results, documents)


## Reranking PDF pages with relevant spans

So far we have reranked short FAQ strings. The rerank endpoint also works on longer documents — here we feed it **all 76 pages of Costco's 2023 10-K annual report (PDF)** and ask which pages describe the CEO transition.

The `relevant_spans` parameter (supported by Rerank v5 and newer models) goes one step further than plain ranking: for each returned document, the model also pinpoints **which passages inside that document** drove the score. Each span comes back as a character range (`start`, `end`) into the original document text plus its own relevance `score`.

In this section we will:
1. Parse a PDF into per-page text chunks
2. Rerank all pages against a question about Costco's CEO transition
3. Display the top pages with a thumbnail, page number, truncated raw text, and the top relevant passages from `relevant_spans`
3. Select the **Open page N in Chrome** button under a card to open the PDF in an external browser directly at that page


[`rerank api doc page`](https://docs.cohere.com/reference/rerank)


In [ ]:
# Parse the PDF into one text chunk per page using pypdfium2 -
# the same open-source PDF engine (pdfium) that Compass uses for parsing
import pypdfium2 as pdfium

pdf_path = "./sample_inputs/COSTCO_2023_10K.pdf"

with pdfium.PdfDocument(pdf_path) as pdf_doc:
    page_texts = []
    for page in pdf_doc:
        textpage = page.get_textpage()
        page_texts.append(textpage.get_text_bounded())  # all text on this page
        textpage.close()  # release the text page handle
        page.close()      # release the page handle

print(f"Parsed {len(page_texts)} pages from {os.path.basename(pdf_path)}\n")
for i in range(3):
    print(f"Page {i + 1}: {len(page_texts[i])} characters — starts with: {page_texts[i][:60].strip()!r}")

In [ ]:
# Rerank all PDF pages and enable relevant_spans so each result
# includes the passages that contributed to its relevance score
query = "Who is retiring as Costco's CEO after twelve years and who is replacing him?"

results = co.rerank(
    model=MODEL,
    query=query,
    documents=page_texts,
    top_n=top_n,
    relevant_spans={"mode": "enabled", "threshold": 0.1, "max_spans": 3},
)

print(f"*** Query: {query}\n")
for rank, result in enumerate(results.results, start=1):
    num_spans = len(result.relevant_spans or [])
    print(f"Rank {rank}: Page {result.index + 1} | score {result.relevance_score:.4f} | relevant spans returned: {num_spans}")

In [ ]:
# Display the top pages: thumbnail, page number, truncated raw text,
# and the top relevant passages returned by relevant_spans.
# Each card gets a button that opens the PDF in an external browser
# (Chrome) directly at that page via the #page= fragment. The click is
# handled by the kernel, which is what allows launching an external app -
# plain HTML links inside the notebook webview cannot do that.
import base64
import html as html_lib
import io
import pathlib
import subprocess
import sys
import ipywidgets as widgets
from IPython.display import HTML, display

MAX_TEXT_CHARS = 400     # truncate raw page text in the output
MAX_PASSAGE_CHARS = 300  # truncate each relevant passage in the output


BROWSER_CANDIDATES = {
    # macOS: direct binary paths (launching the binary preserves the #page
    # fragment; macOS `open -a` strips it via LaunchServices)
    "darwin": [
        "/Applications/Google Chrome.app/Contents/MacOS/Google Chrome",
        "/Applications/Microsoft Edge.app/Contents/MacOS/Microsoft Edge",
        "/Applications/Brave Browser.app/Contents/MacOS/Brave Browser",
    ],
    # Windows: standard install locations for Chrome and Edge
    # (Edge is preinstalled on every Windows machine; Chrome if installed)
    "nt": [
        r"C:\Program Files\Google\Chrome\Application\chrome.exe",
        r"C:\Program Files (x86)\Google\Chrome\Application\chrome.exe",
        r"C:\Program Files (x86)\Microsoft\Edge\Application\msedge.exe",
        r"C:\Program Files\Microsoft\Edge\Application\msedge.exe",
    ],
    "linux": ["/usr/bin/google-chrome", "/usr/bin/microsoft-edge", "/usr/bin/chromium"],
}


def open_pdf_at_page(page_num: int) -> None:
    """Open the PDF in an external browser (Chrome/Edge/Brave) at the given page.

    Chromium-based viewers (Chrome, Edge, Brave) honor the "#page=N" fragment
    in file:// URLs and jump straight to that page. We launch the browser
    executable directly, which works on macOS, Windows and Linux, and opens
    a new tab even when the browser is already running.
    """
    url = pathlib.Path(pdf_path).resolve().as_uri() + f"#page={page_num}"
    candidates = BROWSER_CANDIDATES.get("nt" if os.name == "nt" else sys.platform, [])
    for browser in candidates:
        if pathlib.Path(browser).exists():
            subprocess.Popen([browser, url])
            return
    # Fallback: default handler. On Windows this usually opens Edge and DOES
    # honor the page fragment; on macOS it may ignore it (lands on page 1).
    if sys.platform == "darwin":
        subprocess.run(["open", url], check=False)
    elif os.name == "nt":
        os.startfile(url)  # intentional: Windows default handler
    else:
        subprocess.run(["xdg-open", url], check=False)


pdf = pdfium.PdfDocument(pdf_path)

for rank, result in enumerate(results.results, start=1):
    page_num = result.index + 1
    page_text = page_texts[result.index]

    # Render the page as a PNG thumbnail at half resolution
    page = pdf[page_num - 1]
    bitmap = page.render(scale=0.5)
    thumb_buffer = io.BytesIO()
    bitmap.to_pil().save(thumb_buffer, format="PNG")
    img_b64 = base64.b64encode(thumb_buffer.getvalue()).decode()
    bitmap.close()  # release the bitmap handle
    page.close()    # release the page handle

    truncated_text = page_text[:MAX_TEXT_CHARS].strip()
    if len(page_text) > MAX_TEXT_CHARS:
        truncated_text += " ..."

    spans_html = ""
    for i, span in enumerate((result.relevant_spans or [])[:3], start=1):
        passage = html_lib.escape(page_text[span.start:span.end].strip())
        if len(passage) > MAX_PASSAGE_CHARS:
            passage = passage[:MAX_PASSAGE_CHARS] + " ..."
        spans_html += f'<li><b>Passage {i}</b> (score {span.score:.4f}): "{passage}"</li>'

    card = f"""
    <div style="border:1px solid #ccc; border-radius:8px; padding:12px; margin:10px 0; display:flex; gap:16px;">
      <div style="flex:0 0 auto;">
        <img src="data:image/png;base64,{img_b64}" width="200" style="border:1px solid #ddd;">
        <div style="text-align:center; font-weight:bold;">Page {page_num}</div>
      </div>
      <div style="flex:1 1 auto;">
        <div><b>Rank {rank}</b> - relevance score: {result.relevance_score:.4f}</div>
        <div style="margin:8px 0; color:#555; font-size:0.9em;"><b>Page text (truncated):</b>
          <pre style="white-space:pre-wrap; font-size:0.8em;">{html_lib.escape(truncated_text)}</pre>
        </div>
        <div><b>Top relevant passages</b> (from <code>relevant_spans</code>):</div>
        <ul style="margin-top:4px;">{spans_html}</ul>
      </div>
    </div>
    """
    display(HTML(card))

    # The button click runs in the kernel, so it can launch an external app
    open_btn = widgets.Button(
        description=f"Open page {page_num} in Chrome",
        button_style="primary",
        icon="external-link",
        tooltip=f"Open the PDF at page {page_num} in an external browser",
    )
    open_btn.on_click(lambda b, p=page_num: open_pdf_at_page(p))
    display(open_btn)

pdf.close()

## Conclusion

This tutorial covered how to implement Cohere's rerank endpoint for semantic document ranking, including multilingual support, numeric and temporal reasoning, and negation handling. The rerank endpoint provides a powerful way to improve search relevance by moving beyond simple keyword matching to semantic understanding.

## Important
Once you complete a notebook select **Restart** at the top of VS Code to restart the kernel before moving to the next notebook.